In [1]:
# Necessary libraries 
import pandas as pd
import re
import numpy as np
import seaborn as sns
import nltk
from nltk.corpus import stopwords
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cluster import KMeans
from collections import Counter
from wordcloud import WordCloud
from nltk.stem import WordNetLemmatizer
from sklearn.preprocessing import normalize
from nltk.tokenize import word_tokenize
import nltk

In [2]:
#Load dataset
df = pd.read_csv("customer_support_tickets.csv")
# Remove duplicates
df = df.drop_duplicates(subset=['Ticket Description'], keep='first')
df.reset_index(drop=True, inplace=True)

In [3]:
def preprocess_text(text):
    if not isinstance(text, str):
        return ""
 
    text = text.lower()
    text = re.sub(r'[^a-z\s]', '', text)
    text = re.sub(r'\{.*?\}', '', text)
    text = re.sub(r'http\S+', '', text)
    text = re.sub(r'\b[a-zA-Z]{1,2}\b', '', text)
    words = text.split()
    return ' '.join(words)

df['Processed Description'] = df['Ticket Description'].fillna('').apply(preprocess_text)

df['Processed Description'].head()

0    having issue with the productpurchased please ...
1    having issue with the productpurchased please ...
2    facing problem with productpurchased the produ...
3    having issue with the productpurchased please ...
4    having issue with the productpurchased please ...
Name: Processed Description, dtype: object

In [4]:
nltk.download('punkt')
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('punkt_tab')
stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()
custom_words = {
    'please', 'help', 'assist', 'support', 'thanks', 'thank','soon','mentioned',
    'im', 'ive', 'us','would', 'could', 'need', 'want', 'trying',
    'tried','check', 'checked', 'make', 'made', 'get', 'getting','also',
    'use', 'using', 'used','thing', 'something', 'anything', 'everything',
    'way', 'time','issue', 'problem', 'request',
    'work', 'working', 'fine', 'available', 'recent', 'recently','facing', 'doe',
    'noticed', 'happening', 'started', 'happen','different', 'steps', 'did',
    'regards','already', 'multiple','last','times','followed', 'reviewed',
    'specific', 'possible', 'related','new', 'old','find', 'try', 'trying', 'say', 'mean',
    'name', 'email', 'price', 'one', 'add','note', 'may', 'dont', 'know','sure',
    'changes', 'performed', 'properly','original','like', 'similar','reported','doesnt',
    'sometimes', 'acts', 'works', 'ensure', 'desired', 'action', 'remains', 'life', 'seems',
    'might', 'guide', 'much', 'others','heavily', 'daily', 'task','affecting', 'assistance',
    'hoping','persists','didnt','option', 'perform', 'recommendation', 'information', 'official',
    'solution', 'provide', 'making','user', 'customer','item', 'device','far', 'luck','contact', 'contacted', 'occurring'
}
custom_words_lemma = set([lemmatizer.lemmatize(w) for w in custom_words])
def nlp_clean(text):
    tokens = word_tokenize(text)
    processed_tokens = []
    for word in tokens:
        lemma = lemmatizer.lemmatize(word)
        if lemma not in stop_words and lemma not in custom_words_lemma and len(lemma) > 2:
            processed_tokens.append(lemma)
    return " ".join(processed_tokens)
    
#Apply Cleaning 
#Create the column and basic cleaning
df['Processed Description'] = df['Ticket Description'].fillna('').apply(preprocess_text)

#Refine the text with Lemmatization
df['Processed Description'] = df['Processed Description'].apply(nlp_clean)

# Compare
display(df[['Ticket Description', 'Processed Description']].head())

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\razan\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\razan\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\razan\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\razan\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


,Ticket Description,Processed Description
0,I'm having an issue with the {product_purchase...,productpurchased billing zip code appreciate r...
1,I'm having an issue with the {product_purchase...,productpurchased existing product productpurch...
2,I'm facing a problem with my {product_purchase...,productpurchased productpurchased turning yest...
3,I'm having an issue with the {product_purchase...,productpurchased youre interested love see fee...
4,I'm having an issue with the {product_purchase...,productpurchased seller responsible damage ari...


The preprocessing step removed irrelevant words such as 'please' and 'assist', while retaining key terms related to customer issues. 

In [5]:
# Extract cleaned text to list
documents = df["Processed Description"].tolist()
# Remove empty strings
documents = [doc for doc in documents if len(doc.strip()) >0]
# final output
print(f"Documents ready for modelling: {len(documents)} documents.")
print("\nSample of processed documents:")
for doc in documents[:3]:
    print(f"-{doc}")

Documents ready for modelling: 8077 documents.

Sample of processed documents:
-productpurchased billing zip code appreciate requested website address double address troubleshooting manual
-productpurchased existing product productpurchased intermittent unexpectedly
-productpurchased productpurchased turning yesterday respond really charger came productpurchased charging


In [6]:
df.to_csv("cleaned_tickets.csv", index=False)

print("Saved columns:", df.columns)

Saved columns: Index(['Ticket ID', 'Customer Name', 'Customer Email', 'Customer Age',
       'Customer Gender', 'Product Purchased', 'Date of Purchase',
       'Ticket Type', 'Ticket Subject', 'Ticket Description', 'Ticket Status',
       'Resolution', 'Ticket Priority', 'Ticket Channel',
       'First Response Time', 'Time to Resolution',
       'Customer Satisfaction Rating', 'Processed Description'],
      dtype='object')
